1. Подготовка данных.
Сканируем папку `planes/`, читаем истинные метки из имен файлов (`_x` — военный, `_y` — гражданский) и создаем базовую таблицу results_both_models

In [1]:
import pandas as pd
import os

# --- Настройки ---
IMAGE_DIR = "planes"
CSV_PATH = "results_both_models.csv"

def get_true_label(filename):
    """_x -> M (военный), _y -> C (гражданский)"""
    base = os.path.splitext(filename.lower())[0]
    if base.endswith("_x"):
        return "M"
    elif base.endswith("_y"):
        return "C"
    return None

data = []
skipped = []

for filename in sorted(os.listdir(IMAGE_DIR)):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        label = get_true_label(filename)
        if label is None:
            skipped.append(filename)
            continue
        data.append({
            "image_path": os.path.join(IMAGE_DIR, filename),
            "true_label": label
        })

df = pd.DataFrame(data)
df.to_csv(CSV_PATH, index=False)

print(f"✅ Создан файл: {CSV_PATH}")
print(f"✅ Строк: {len(df)}")
print(f"   Столбцы: {df.columns.tolist()}")
if skipped:
    print(f"⚠️ Пропущено без метки: {len(skipped)}")
    for f in skipped:
        print(f"   - {f}")
print("\nРаспределение истинных меток:")
print(df['true_label'].value_counts())

✅ Создан файл: results_both_models.csv
✅ Строк: 30
   Столбцы: ['image_path', 'true_label']

Распределение истинных меток:
true_label
M    15
C    15
Name: count, dtype: int64


2. Работа модели llava
Отправляем каждое изображение в локальную модель `llava` через Ollama. Модель должна ответить одной буквой: `M` (военный) или `C` (гражданский). Результат заносится в таблицу results_both_models.

In [15]:
import ollama
import pandas as pd
import os
import time

# --- Настройки ---
MODEL_1 = "llava"
CSV_PATH = "results_both_models.csv"

# --- df из памяти или из файла ---
try:
    _ = df
    print(f"Строк: {len(df)}")
except NameError:
    print(f"Загружаем из {CSV_PATH}...")
    df = pd.read_csv(CSV_PATH)
    print(f"Строк: {len(df)}")

# --- Функция предсказания (M / C) ---
def predict_plane_type(image_path, model_name):
    if not os.path.exists(image_path):
        return None
    try:
        response = ollama.chat(
            model=model_name,
            messages=[{
                'role': 'user',
                'content': (
                    'Look at the aircraft in the image. '
                    'Is it military or civilian? '
                    'Answer with exactly one letter: "M" for military, "C" for civilian. '
                    'Do not write anything else.'
                ),
                'images': [image_path]
            }],
            options={'temperature': 0}
        )
        answer = response['message']['content'].strip().upper()

        if answer == "M":
            return "M"
        if answer == "C":
            return "C"
        if "MILIT" in answer or "FIGHT" in answer or "COMBAT" in answer or "ВОЕН" in answer:
            return "M"
        if "CIVIL" in answer or "PASSENG" in answer or "COMMERCIAL" in answer or "ГРАЖ" in answer:
            return "C"

        return answer  # непонятный ответ — оставляем для разбора
    except Exception as e:
        print(f"\nОшибка на {image_path}: {e}")
        return None

# --- Прогон ---
predictions_1 = []
print(f"\nЗапуск {MODEL_1} по {len(df)} картинкам...\n")
start_time = time.time()

for i, row in df.iterrows():
    pred = predict_plane_type(row['image_path'], MODEL_1)
    predictions_1.append(pred)
    if (i + 1) % 5 == 0 or (i + 1) == len(df):
        elapsed = time.time() - start_time
        print(f"{i+1}/{len(df)} | {elapsed:.0f} сек | Последний: {pred}")

df[f'pred_{MODEL_1}'] = predictions_1
df.to_csv(CSV_PATH, index=False)
print(f"\nГотово! pred_{MODEL_1} сохранен в {CSV_PATH}")
print("\nРаспределение ответов:")
print(df[f'pred_{MODEL_1}'].value_counts(dropna=False))

df в памяти. Строк: 30

Запуск llava по 30 картинкам...

5/30 | 3 сек | Последний: M
10/30 | 5 сек | Последний: M
15/30 | 6 сек | Последний: M
20/30 | 7 сек | Последний: C
25/30 | 8 сек | Последний: C
30/30 | 10 сек | Последний: M

Готово! pred_llava сохранен в results_both_models.csv

Распределение ответов:
pred_llava
C    16
M    14
Name: count, dtype: int64


3. Работа модели minicpm-v4.5
Отправляем каждое изображение в локальную модель `minicpm-v4.5` через Ollama. Модель должна ответить одной буквой: `M` (военный) или `C` (гражданский). Результат заносится в таблицу results_both_models.

In [16]:
import ollama
import pandas as pd
import os
import time

# --- Настройки ---
MODEL_2 = "minicpm-v4.5"
CSV_PATH = "results_both_models.csv"

# --- df из памяти или из файла ---
try:
    _ = df
    print(f"Строк: {len(df)}")
except NameError:
    print(f"Загружаем из {CSV_PATH}...")
    df = pd.read_csv(CSV_PATH)
    print(f"Строк: {len(df)}")

# --- Та же функция (блок автономный) ---
def predict_plane_type(image_path, model_name):
    if not os.path.exists(image_path):
        return None
    try:
        response = ollama.chat(
            model=model_name,
            messages=[{
                'role': 'user',
                'content': (
                    'Look at the aircraft in the image. '
                    'Is it military or civilian? '
                    'Answer with exactly one letter: "M" for military, "C" for civilian. '
                    'Do not write anything else.'
                ),
                'images': [image_path]
            }],
            options={'temperature': 0}
        )
        answer = response['message']['content'].strip().upper()

        if answer == "M":
            return "M"
        if answer == "C":
            return "C"
        if "MILIT" in answer or "FIGHT" in answer or "COMBAT" in answer or "ВОЕН" in answer:
            return "M"
        if "CIVIL" in answer or "PASSENG" in answer or "COMMERCIAL" in answer or "ГРАЖ" in answer:
            return "C"

        return answer
    except Exception as e:
        print(f"\nОшибка на {image_path}: {e}")
        return None

# --- Прогон ---
predictions_2 = []
print(f"\nЗапуск {MODEL_2} по {len(df)} картинкам...\n")
start_time = time.time()

for i, row in df.iterrows():
    pred = predict_plane_type(row['image_path'], MODEL_2)
    predictions_2.append(pred)
    if (i + 1) % 5 == 0 or (i + 1) == len(df):
        elapsed = time.time() - start_time
        print(f"{i+1}/{len(df)} | {elapsed:.0f} сек | Последний: {pred}")

df[f'pred_{MODEL_2}'] = predictions_2
df.to_csv(CSV_PATH, index=False)
print(f"\nГотово! pred_{MODEL_2} сохранен в {CSV_PATH}")
print("\nРаспределение ответов:")
print(df[f'pred_{MODEL_2}'].value_counts(dropna=False))

Строк: 30

Запуск minicpm-v4.5 по 30 картинкам...

5/30 | 6 сек | Последний: M
10/30 | 8 сек | Последний: M
15/30 | 11 сек | Последний: C
20/30 | 14 сек | Последний: C
25/30 | 18 сек | Последний: M
30/30 | 20 сек | Последний: M

Готово! pred_minicpm-v4.5 сохранен в results_both_models.csv

Распределение ответов:
pred_minicpm-v4.5
M    18
C    12
Name: count, dtype: int64


In [ ]:
4. Метрики и сравнение моделей

Считаем Accuracy, Precision, Recall и F1-score для каждой модели, выводим список ошибок и сохраняем данные в `final_report.txt`.

In [18]:
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- Загружаем файл ---
df = pd.read_csv("results_both_models.csv")

# Короткое имя файла для аккуратной таблицы
df['file_name'] = df['image_path'].apply(lambda x: os.path.basename(str(x).replace('\\', '/')))

# --- Функция метрик ---
def get_metrics(true_labels, pred_labels, model_name):
    clean_preds = pred_labels.astype(str).str.strip().str.upper()
    mask = clean_preds.isin(["M", "C"])
    y_true = true_labels[mask]
    y_pred = clean_preds[mask]
    if len(y_true) == 0:
        return {"Model": model_name, "Valid": f"0/{len(true_labels)}",
                "Accuracy": None, "Precision": None, "Recall": None, "F1": None}
    return {
        "Model": model_name,
        "Valid": f"{len(y_true)}/{len(true_labels)}",
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred, pos_label='M', average='binary', zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, pos_label='M', average='binary', zero_division=0), 4),
        "F1": round(f1_score(y_true, y_pred, pos_label='M', average='binary', zero_division=0), 4),
    }

# --- Считаем для всех моделей (убираем pred_ из названия) ---
pred_cols = [c for c in df.columns if c.startswith("pred_")]
comparison = pd.DataFrame([
    get_metrics(df['true_label'], df[c], c.replace("pred_", "")) 
    for c in pred_cols
])

print("=== Сравнение моделей ===\n")
print(comparison.to_string(index=False))

# --- Ошибки ---
print("\n" + "="*60)
for c in pred_cols:
    errs = df[df[c].astype(str).str.upper() != df['true_label'].astype(str).str.upper()][['file_name', 'true_label', c]]
    model_name = c.replace("pred_", "")
    print(f"\nОшибки {model_name} ({len(errs)} шт.):")
    if len(errs) == 0:
        print("  (нет ошибок)")
    else:
        print(errs.rename(columns={c: model_name}).to_string(index=False))

# --- Сохраняем отчет ---
with open("final_report.txt", "w", encoding="utf-8") as f:
    f.write("=== Сравнение моделей ===\n\n")
    f.write(comparison.to_string(index=False))
    f.write("\n\n")
    for c in pred_cols:
        errs = df[df[c].astype(str).str.upper() != df['true_label'].astype(str).str.upper()][['file_name', 'true_label', c]]
        model_name = c.replace("pred_", "")
        f.write(f"\n=== Ошибки {model_name} ({len(errs)} шт.) ===\n")
        if len(errs) == 0:
            f.write("  (нет ошибок)\n")
        else:
            f.write(errs.rename(columns={c: model_name}).to_string(index=False))
        f.write("\n")

print("\n✅ Отчет сохранен в final_report.txt")

=== Сравнение моделей ===

       Model Valid  Accuracy  Precision  Recall     F1
       llava 30/30       0.9     0.9286  0.8667 0.8966
minicpm-v4.5 30/30       0.9     0.8333  1.0000 0.9091


Ошибки llava (3 шт.):
                                                             file_name true_label llava
                                112f18e12f95a6aec6281f33fcead800_x.jpg          M     C
1641063729_19-hdpic-club-p-samoleti-pervoi-mirovoi-voini-foto-60_x.jpg          M     C
                                                      efae6a9e77_y.jpg          C     M

Ошибки minicpm-v4.5 (3 шт.):
           file_name true_label minicpm-v4.5
         i (4)_y.jpg          C            M
scale_1200 (3)_y.jpg          C            M
       stal3-2_y.jpg          C            M

✅ Отчет сохранен в final_report.txt
